# gramlab — chat
Ask the retrieval-conditioned 25M model questions. **Interactive use only** (widgets don't run in Save & Run All).
Setup once: *Add Input → Your Work → Notebooks → gramlab* (latest version), **Accelerator: None** (CPU is enough, costs no GPU quota).
Then: **Run All**, wait ~40 s, type in the box.

In [ ]:
import os, sys, glob, subprocess, torch, ipywidgets as w
from IPython.display import display
if not os.path.exists('/kaggle/working/gramlab'):
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/furevych/gramlab.git', '/kaggle/working/gramlab'], check=True)
sys.path.insert(0, '/kaggle/working/gramlab')
cands = sorted(glob.glob('/kaggle/input/**/work/runs/small_wiki_rag/ckpt.pt', recursive=True),
               key=lambda p: -torch.load(p, map_location='cpu', weights_only=False)['step'])
if not cands:
    raise SystemExit('Add the gramlab notebook output as an input (right panel -> Input -> Add Input -> Your Work)')
os.chdir(cands[0].split('/work/')[0])
from gramlab.ckpt import load_checkpoint
from gramlab.tokenizer import USER, ASSISTANT, END, EOS
from gramlab.retrieval import BM25Index, format_rag_user
model, tok, state = load_checkpoint('work/runs/small_wiki_rag/ckpt.pt', 'cpu')
index = BM25Index.load('work/bm25')
print(f"writer step {state['step']}, BM25 over {len(index):,} passages, from {os.getcwd()}")

def ask(q, k=1, show_passage=True, temperature=0.3):
    p = index.retrieve(q, k=k, max_words=90) or [{'title': '(no match)', 'text': 'No passage found.', 'score': 0}]
    ids = [USER] + tok.encode(format_rag_user(p, q)) + [END, ASSISTANT]
    out = model.generate(torch.tensor([ids[-(model.ctx - 1):]]), max_new_tokens=80, temperature=temperature,
                         top_k=40, top_p=0.9, repetition_penalty=1.2, stop_ids=(END, EOS))
    if show_passage:
        for x in p: print(f"  [read: {x['title']}]")
    return tok.decode(out[0, len(ids):].tolist())

box = w.Text(placeholder='ask about a place, a person, a thing...', layout=w.Layout(width='70%'), continuous_update=False)
btn, out = w.Button(description='Send'), w.Output()
def on_send(_=None):
    q = box.value.strip()
    if not q: return
    box.value = ''
    with out:
        print('you>  ', q); print('model>', ask(q)); print()
btn.on_click(on_send)
box.observe(lambda ch: on_send() if ch['name'] == 'value' and ch['new'] else None, names='value')
display(w.HBox([box, btn]), out)

In [ ]:
# or without the widget: one question per cell
ask("Who was Winston Churchill?")